# Brain tumour segmentation on MRI with a U-Net (PyTorch)

A 2D U-Net (5 encoder levels, base 32 filters, batch normalisation, dropout in the bottleneck) for tumour segmentation in MRI slices, with Monte Carlo dropout uncertainty.

What the notebook does:

* **Evaluation.** Dice is computed per image and also pooled over all test pixels. Slices with a tumour and slices without one are reported separately.
* **Masks.** Masks are resized with nearest-neighbour interpolation so they stay binary.
* **Data pipeline.** The data is loaded once into memory, so training does not wait on disk or Google Drive.
* **Uncertainty.** Monte Carlo dropout gives a per-pixel uncertainty map and a per-image uncertainty score. The notebook checks whether that score flags poor segmentations, and how Dice changes if the model abstains on its most uncertain cases.

In [ ]:
# Setup (Colab: Runtime > Change runtime type > GPU)
import os, sys, json, random, time, shutil
try:
    import pycocotools  # noqa: F401
except ImportError:
    !pip -q install pycocotools

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pycocotools.coco import COCO
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Configuration
DATA_ROOT = os.environ.get("DATA_ROOT", "/content/drive/MyDrive/Brain-MRI-segmentation")
if not os.path.exists(os.path.join(DATA_ROOT, "train", "_annotations.coco.json")):
    # Fall back to downloading the Kaggle dataset (public, no key needed in Colab)
    import glob, kagglehub
    _p = kagglehub.dataset_download("pkdarabi/brain-tumor-image-dataset-semantic-segmentation")
    DATA_ROOT = os.path.dirname(os.path.dirname(sorted(glob.glob(_p + "/**/_annotations.coco.json", recursive=True))[0]))
print("DATA_ROOT:", DATA_ROOT)
OUT_DIR   = os.environ.get("OUT_DIR", "outputs")
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"   # tiny run used only to check the code

IMG_SIZE   = 256
BATCH_SIZE = 16
EPOCHS     = int(os.environ.get("EPOCHS", 2 if SMOKE_TEST else 60))
LR         = float(os.environ.get("LR", 1e-4))
PATIENCE   = 10          # early stopping on validation Dice
BASE_FILTERS = 8 if SMOKE_TEST else 32
MC_SAMPLES = 3 if SMOKE_TEST else 20
THRESH     = 0.5
SEED       = 42

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)
print("device:", device)

## Data
The COCO annotations are turned into binary tumour masks. Images are read as greyscale; masks are resized with nearest-neighbour interpolation.

In [ ]:
def load_split(split):
    folder = os.path.join(DATA_ROOT, split)
    coco = COCO(os.path.join(folder, "_annotations.coco.json"))
    ids = sorted(coco.getImgIds())
    imgs = np.zeros((len(ids), IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    masks = np.zeros((len(ids), IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for k, img_id in enumerate(ids):
        info = coco.loadImgs(img_id)[0]
        img = Image.open(os.path.join(folder, info["file_name"])).convert("L")
        m = np.zeros((info["height"], info["width"]), dtype=np.uint8)
        for ann in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
            m = np.maximum(m, coco.annToMask(ann))
        imgs[k] = np.asarray(img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR))
        masks[k] = np.asarray(Image.fromarray(m).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
    return imgs, masks

t0 = time.time()
data = {s: load_split(s) for s in ["train", "valid", "test"]}
for s, (x, y) in data.items():
    has_tumour = (y.reshape(len(y), -1).max(1) > 0)
    print(f"{s:5s}: {len(x):4d} images | with tumour: {has_tumour.sum():4d} | "
          f"tumour pixels: {100 * y.mean():.2f}%")
print(f"loaded in {time.time() - t0:.0f}s")

In [ ]:
class SliceDataset(Dataset):
    def __init__(self, imgs, masks):
        self.x = torch.from_numpy(imgs).float().div(255.).unsqueeze(1)   # (N,1,H,W)
        self.y = torch.from_numpy(masks).float().unsqueeze(1)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, i):
        return self.x[i], self.y[i]

loaders = {
    s: DataLoader(SliceDataset(*data[s]), batch_size=BATCH_SIZE, shuffle=(s == "train"),
                  num_workers=2, pin_memory=(device.type == "cuda"), drop_last=False)
    for s in data
}

## Augmentation (on the GPU, per batch)
Flips, small rotations, elastic deformation, brightness/contrast and Gaussian noise. Geometric transforms move image and mask together; the mask uses nearest-neighbour sampling.

In [ ]:
def gaussian_kernel(sigma, device):
    r = int(3 * sigma)
    t = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    k = torch.exp(-t ** 2 / (2 * sigma ** 2))
    return k / k.sum()

def augment(x, y, p=0.5, max_rot_deg=20, elastic_alpha=0.03, elastic_sigma=8):
    B, _, H, W = x.shape
    dev = x.device
    # flips
    flip_h = torch.rand(B, device=dev) < p
    flip_v = torch.rand(B, device=dev) < p
    x = torch.where(flip_h[:, None, None, None], x.flip(3), x)
    y = torch.where(flip_h[:, None, None, None], y.flip(3), y)
    x = torch.where(flip_v[:, None, None, None], x.flip(2), x)
    y = torch.where(flip_v[:, None, None, None], y.flip(2), y)
    # rotation (applied to a random half of the batch)
    ang = (torch.rand(B, device=dev) * 2 - 1) * np.deg2rad(max_rot_deg)
    ang = ang * (torch.rand(B, device=dev) < p)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.zeros(B, 2, 3, device=dev)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 1, 0], theta[:, 1, 1] = cos, -sin, sin, cos
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    # elastic deformation: smooth random displacement field
    use_el = (torch.rand(B, device=dev) < p)[:, None, None, None]
    disp = torch.rand(B, 2, H, W, device=dev) * 2 - 1
    k = gaussian_kernel(elastic_sigma, dev)
    disp = F.conv2d(disp.reshape(-1, 1, H, W), k.view(1, 1, 1, -1), padding=(0, len(k) // 2))
    disp = F.conv2d(disp, k.view(1, 1, -1, 1), padding=(len(k) // 2, 0)).reshape(B, 2, H, W)
    disp = disp / (disp.abs().amax(dim=(2, 3), keepdim=True) + 1e-8) * elastic_alpha
    grid = grid + (disp.permute(0, 2, 3, 1) * use_el.squeeze(1)[..., None])
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    y = F.grid_sample(y, grid, mode="nearest", padding_mode="zeros", align_corners=False)
    # intensity
    bright = (torch.rand(B, 1, 1, 1, device=dev) * 0.2 - 0.1) * (torch.rand(B, 1, 1, 1, device=dev) < p)
    contr = 1 + (torch.rand(B, 1, 1, 1, device=dev) * 0.2 - 0.1) * (torch.rand(B, 1, 1, 1, device=dev) < p)
    mean = x.mean(dim=(2, 3), keepdim=True)
    x = (x - mean) * contr + mean + bright
    noise = torch.randn_like(x) * 0.02 * (torch.rand(B, 1, 1, 1, device=dev) < p)
    return (x + noise).clamp(0, 1), y

In [ ]:
# Check a few augmented samples
xb, yb = next(iter(loaders["train"]))
xa, ya = augment(xb.to(device), yb.to(device))
fig, ax = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    ax[0, i].imshow(xa[i, 0].cpu(), cmap="gray"); ax[0, i].contour(ya[i, 0].cpu(), colors="r", linewidths=0.8)
    ax[1, i].imshow(xb[i, 0], cmap="gray"); ax[1, i].contour(yb[i, 0], colors="r", linewidths=0.8)
    ax[0, i].axis("off"); ax[1, i].axis("off")
ax[0, 0].set_title("augmented"); ax[1, 0].set_title("original")
plt.tight_layout(); plt.show()

## U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, c_in, c_out, drop=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )
        self.drop = nn.Dropout2d(drop) if drop > 0 else nn.Identity()
    def forward(self, x):
        return self.drop(self.block(x))

class UNet(nn.Module):
    """5 down-sampling levels + bottleneck."""
    def __init__(self, in_ch=1, base=32, depth=5, bottleneck_drop=0.4, decoder_drop=0.1):
        super().__init__()
        chs = [base * 2 ** i for i in range(depth)]           # 32 .. 512
        self.enc = nn.ModuleList()
        c = in_ch
        for ch in chs:
            self.enc.append(DoubleConv(c, ch)); c = ch
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(c, c * 2, drop=bottleneck_drop)   # 1024
        c = c * 2
        self.up, self.dec = nn.ModuleList(), nn.ModuleList()
        for ch in reversed(chs):
            self.up.append(nn.ConvTranspose2d(c, ch, 2, stride=2))
            self.dec.append(DoubleConv(ch * 2, ch, drop=decoder_drop)); c = ch
        self.head = nn.Conv2d(c, 1, 1)
    def forward(self, x):
        skips = []
        for enc in self.enc:
            x = enc(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for up, dec, s in zip(self.up, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], dim=1))
        return self.head(x)                                  # logits

model = UNet(base=BASE_FILTERS).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

## Loss and metrics
Loss: 0.6 soft Dice + 0.4 binary cross-entropy (as before). Metrics are computed **per image**, plus a pooled Dice over all test pixels.

In [ ]:
def hybrid_loss(logits, target, w_dice=0.6, eps=1.0):
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(1, 2, 3))
    denom = prob.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - (2 * inter + eps) / (denom + eps)
    bce = F.binary_cross_entropy_with_logits(logits, target)
    return w_dice * dice.mean() + (1 - w_dice) * bce

def confusion_per_image(pred, target):
    """pred, target: binary tensors (B,1,H,W) -> TP, FP, FN, TN per image."""
    pred, target = pred.bool(), target.bool()
    tp = (pred & target).sum(dim=(1, 2, 3))
    fp = (pred & ~target).sum(dim=(1, 2, 3))
    fn = (~pred & target).sum(dim=(1, 2, 3))
    tn = (~pred & ~target).sum(dim=(1, 2, 3))
    return torch.stack([tp, fp, fn, tn], 1).cpu().numpy().astype(np.int64)

def summarise(cm):
    """cm: (N,4) TP,FP,FN,TN per image. Dice/IoU are defined on images with a tumour."""
    tp, fp, fn, tn = cm.T
    has_t = (tp + fn) > 0
    dice = np.where(has_t, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan)
    iou = np.where(has_t, tp / np.maximum(tp + fp + fn, 1), np.nan)
    sens = np.where(has_t, tp / np.maximum(tp + fn, 1), np.nan)
    prec = np.where((tp + fp) > 0, tp / np.maximum(tp + fp, 1), np.nan)
    T, P, N_, _ = cm.sum(0)[[0, 1, 2, 3]]
    return {
        "n_images": int(len(cm)),
        "n_with_tumour": int(has_t.sum()),
        "dice_per_image_mean": float(np.nanmean(dice)),
        "dice_per_image_median": float(np.nanmedian(dice)),
        "iou_per_image_mean": float(np.nanmean(iou)),
        "sensitivity_per_image_mean": float(np.nanmean(sens)),
        "precision_per_image_mean": float(np.nanmean(prec)) if np.isfinite(prec).any() else float("nan"),
        "dice_pooled": float(2 * T / max(2 * T + P + N_, 1)),
        "sensitivity_pooled": float(T / max(T + N_, 1)),
        "precision_pooled": float(T / max(T + P, 1)),
        # slices without tumour: how often does the model predict tumour anyway?
        "false_alarm_rate_empty_slices": float(((tp + fp)[~has_t] > 0).mean()) if (~has_t).any() else float("nan"),
        "missed_tumour_rate": float((tp[has_t] == 0).mean()) if has_t.any() else float("nan"),
    }

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    cms, losses = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        losses.append(hybrid_loss(logits, y).item() * len(x))
        cms.append(confusion_per_image(torch.sigmoid(logits) > THRESH, y))
    cm = np.concatenate(cms)
    out = summarise(cm)
    out["loss"] = float(np.sum(losses) / len(cm))
    return out, cm

## Training
Adam, ReduceLROnPlateau, and early stopping on the mean per-image validation Dice. The best checkpoint is saved.

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.1, patience=4, min_lr=1e-6)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
ckpt_path = os.path.join(OUT_DIR, "best_unet.pt")

history, best, bad_epochs = [], -1.0, 0
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    for x, y in loaders["train"]:
        x, y = augment(x.to(device, non_blocking=True), y.to(device, non_blocking=True))
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss = hybrid_loss(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run += loss.item() * len(x)
    train_loss = run / len(loaders["train"].dataset)
    val, _ = evaluate(model, loaders["valid"])
    sched.step(val["dice_per_image_mean"])
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val["loss"],
                    "val_dice": val["dice_per_image_mean"], "val_sens": val["sensitivity_per_image_mean"],
                    "lr": opt.param_groups[0]["lr"]})
    print(f"epoch {epoch:3d} | train loss {train_loss:.4f} | val loss {val['loss']:.4f} | "
          f"val Dice {val['dice_per_image_mean']:.4f} | val sens {val['sensitivity_per_image_mean']:.4f} | "
          f"{time.time() - t0:.0f}s")
    if val["dice_per_image_mean"] > best:
        best, bad_epochs = val["dice_per_image_mean"], 0
        torch.save(model.state_dict(), ckpt_path)
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("early stopping"); break

model.load_state_dict(torch.load(ckpt_path, map_location=device))

In [ ]:
h = {k: [r[k] for r in history] for k in history[0]}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["epoch"], h["train_loss"], label="train"); ax[0].plot(h["epoch"], h["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(h["epoch"], h["val_dice"], label="Dice"); ax[1].plot(h["epoch"], h["val_sens"], label="sensitivity")
ax[1].set_title("Validation (per image)"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "training_curves.png"), dpi=150); plt.show()

## Test-set results

In [ ]:
test_res, test_cm = evaluate(model, loaders["test"])
for k, v in test_res.items():
    print(f"{k:32s} {v:.4f}" if isinstance(v, float) else f"{k:32s} {v}")

## Uncertainty with Monte Carlo dropout
Dropout stays active at test time (batch norm stays in eval mode) and each image is passed through the model `MC_SAMPLES` times. The mean gives the prediction, and the binary entropy of the mean gives a per-pixel uncertainty map.

The per-image uncertainty score is the mean entropy inside the predicted tumour region plus a one-pixel border; for images with no predicted tumour it is the mean entropy over the whole image. The next cells check whether a high score goes with a low Dice, and what happens to Dice if the model abstains on its most uncertain cases.

In [ ]:
def enable_mc_dropout(m):
    m.eval()
    for mod in m.modules():
        if isinstance(mod, (nn.Dropout, nn.Dropout2d)):
            mod.train()

@torch.no_grad()
def mc_predict(model, loader, T=MC_SAMPLES):
    enable_mc_dropout(model)
    means, ents, ys, xs = [], [], [], []
    for x, y in loader:
        x = x.to(device)
        p = torch.stack([torch.sigmoid(model(x)) for _ in range(T)]).mean(0)
        pc = p.clamp(1e-6, 1 - 1e-6)
        ent = -(pc * pc.log() + (1 - pc) * (1 - pc).log()) / np.log(2)    # bits, 0..1
        means.append(p.cpu()); ents.append(ent.cpu()); ys.append(y); xs.append(x.cpu())
    model.eval()
    return torch.cat(xs), torch.cat(ys), torch.cat(means), torch.cat(ents)

x_te, y_te, p_te, ent_te = mc_predict(model, loaders["test"])
pred_te = p_te > THRESH
mc_cm = confusion_per_image(pred_te, y_te)
mc_res = summarise(mc_cm)

def image_uncertainty(pred, ent):
    region = F.max_pool2d(pred.float(), 3, stride=1, padding=1) > 0      # predicted region + 1 px border
    scores = []
    for r, e in zip(region, ent):
        scores.append(e[r].mean().item() if r.any() else e.mean().item())
    return np.array(scores)

unc = image_uncertainty(pred_te, ent_te)
tp, fp, fn, tn = mc_cm.T
has_t = (tp + fn) > 0
dice_img = np.where(has_t, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan)

def spearman(a, b):
    ra, rb = np.argsort(np.argsort(a)), np.argsort(np.argsort(b))
    return float(np.corrcoef(ra, rb)[0, 1])

rho = spearman(unc[has_t], dice_img[has_t]) if has_t.sum() > 2 else float("nan")
print(f"MC-dropout mean per-image Dice (tumour slices): {mc_res['dice_per_image_mean']:.4f}")
print(f"Spearman correlation, uncertainty vs Dice (tumour slices): {rho:.3f}  (negative = uncertainty flags bad cases)")

In [ ]:
# Risk-coverage: abstain on the most uncertain images, measure Dice on what is kept
idx = np.where(has_t)[0]
order = idx[np.argsort(unc[idx])]                 # most certain first
coverages = np.linspace(0.1, 1.0, 10)
kept_dice = [np.nanmean(dice_img[order[:max(1, int(round(c * len(order))))]]) for c in coverages]
rng = np.random.default_rng(SEED)
rand_dice = [np.mean([np.nanmean(dice_img[rng.permutation(idx)[:max(1, int(round(c * len(idx))))]])
                      for _ in range(200)]) for c in coverages]

plt.figure(figsize=(6, 4))
plt.plot(coverages, kept_dice, "o-", label="abstain on most uncertain")
plt.plot(coverages, rand_dice, "--", label="abstain at random")
plt.xlabel("coverage (fraction of tumour slices kept)"); plt.ylabel("mean Dice on kept slices")
plt.title("Risk-coverage on the test set"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "risk_coverage.png"), dpi=150); plt.show()

In [ ]:
# Examples: best, median and worst test cases, with the uncertainty map
def show_cases(ids, title):
    fig, ax = plt.subplots(len(ids), 4, figsize=(13, 3.2 * len(ids)), squeeze=False)
    for r, i in enumerate(ids):
        ax[r, 0].imshow(x_te[i, 0], cmap="gray"); ax[r, 0].set_title("MRI")
        ax[r, 1].imshow(y_te[i, 0], cmap="gray"); ax[r, 1].set_title("ground truth")
        ax[r, 2].imshow(pred_te[i, 0], cmap="gray"); ax[r, 2].set_title(f"prediction, Dice {dice_img[i]:.2f}")
        im = ax[r, 3].imshow(ent_te[i, 0], cmap="magma", vmin=0, vmax=1); ax[r, 3].set_title(f"uncertainty, score {unc[i]:.2f}")
        for a in ax[r]: a.axis("off")
    fig.colorbar(im, ax=ax[:, 3].tolist(), shrink=0.6)
    fig.suptitle(title); plt.savefig(os.path.join(OUT_DIR, title.replace(" ", "_") + ".png"), dpi=150, bbox_inches="tight"); plt.show()

ranked = idx[np.argsort(dice_img[idx])]
if len(ranked):
    show_cases([ranked[-1], ranked[len(ranked) // 2], ranked[0]], "best median worst")

In [ ]:
# Save results for the README
results = {
    "test": test_res,
    "test_mc_dropout": mc_res,
    "uncertainty_vs_dice_spearman": rho,
    "risk_coverage": {f"{c:.1f}": float(d) for c, d in zip(coverages, kept_dice)},
    "config": {"img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "lr": LR, "epochs_run": len(history),
               "base_filters": BASE_FILTERS, "mc_samples": MC_SAMPLES, "threshold": THRESH, "seed": SEED},
}
def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, float) and not np.isfinite(o): return None
    return o
results = _clean(results)
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))